# ANZSRC FoR 2020: великі моделі на Kaggle

Повне донавчання двох великих моделей на очищених мітках, кожна на своїй T4 (паралельно):
**DeBERTa-v3-large** і **ModernBERT-large** (якщо котрась не запуститься, замість неї — RoBERTa-large).
Той самий скрипт `03_train_transformer.py`, що й на ноутбуці.

Перед запуском (праворуч, панель *Session options* / *Settings*):
1. **Add Input** → твій приватний набір даних із `kaggle_upload.zip`;
2. **Accelerator** → **GPU T4 x2**;
3. **Internet** → **On** (щоб завантажити моделі).

Потім **Save Version → Save & Run All (Commit)**. Браузер можна закрити: усе йде у фоні, ~7–9 годин,
а навчання саме зупиниться не пізніше ніж за 11 годин (Kaggle обриває запуск після 12).
Коли закінчиться: відкрий цю версію ноутбука → **Output** → завантаж `deberta_large_clean.zip`,
`modernbert_large_clean.zip` і `reports_kaggle.zip`. На ноутбуці: `run_after_kaggle.bat`.

In [ ]:
# Налаштування. Для звичайного запуску нічого не змінюй.
from pathlib import Path
INPUT, OUTPUT = Path('/kaggle/input'), Path('/kaggle/working')
JOBS = [('microsoft/deberta-v3-large', 3, '1.5e-5'),     # модель, епохи, швидкість навчання
        ('answerdotai/ModernBERT-large', 3, '2e-5'),       # одна модель на одну відеокарту; якщо котрась
        ('FacebookAI/roberta-large', 3, '1.5e-5')]         # не пройде перевірку, її місце займе наступна
BUDGET_H = 11.0                                   # години на навчання (Kaggle обриває запуск через 12)
TRAIN_ARGS = ['--train', 'train_clean', '--batch', '16', '--no-state']
DEVICE = 'cuda'

In [ ]:
# 1. Бібліотеки: transformers тієї самої версії, що на ноутбуці (моделі потім відкриються там без проблем)
import os, subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==5.22.1', 'sentencepiece', 'protobuf'],
               check=True)
import torch
N_GPU = torch.cuda.device_count() if DEVICE == 'cuda' else 1
print('torch', torch.__version__)
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {p.name}, {p.total_memory / 2**30:.0f} ГБ')
assert N_GPU > 0, 'Немає відеокарти: у Settings → Accelerator вибери GPU T4 x2 і запусти знову'
if N_GPU < 2:
    print('Відеокарта одна, тож навчатиметься лише одна модель. Для двох моделей вибери GPU T4 x2.')

In [ ]:
# 2. Скрипти й дані з набору даних (kaggle_upload.zip; Kaggle зазвичай сам його розпаковує)
import glob, shutil, zipfile
W = OUTPUT / 'training'
(W / 'data').mkdir(parents=True, exist_ok=True)
NEED = {'common.py': W, '03_train_transformer.py': W, 'train_clean.parquet': W / 'data',
        'val.parquet': W / 'data', 'test.parquet': W / 'data', 'labels.json': W / 'data'}
found = {Path(p).name: Path(p) for p in glob.glob(str(INPUT / '**' / '*'), recursive=True) if os.path.isfile(p)}
for z in [p for p in found.values() if p.suffix == '.zip']:            # якщо архів не розпаковано
    with zipfile.ZipFile(z) as zf:
        for info in zf.infolist():
            name = Path(info.filename).name
            if name in NEED and name not in found:
                with zf.open(info) as src, open(NEED[name] / name, 'wb') as dst:
                    shutil.copyfileobj(src, dst)
                found[name] = NEED[name] / name
missing = [n for n in NEED if n not in found]
assert not missing, f'У доданих даних немає: {missing}. Додай (Add Input) набір даних із kaggle_upload.zip.'
for n, d in NEED.items():
    if found[n] != d / n:
        shutil.copy(found[n], d / n)
print('Файли на місці:', ', '.join(sorted(NEED)))

In [ ]:
# 3. Коротка перевірка кожної моделі (завантаження, токенізатор, крок навчання), потім запуск навчання
import time

def preflight(model):
    code = f'''
import torch
from transformers import AutoModel, AutoTokenizer
tok = AutoTokenizer.from_pretrained({model!r})
m = AutoModel.from_pretrained({model!r}).to({DEVICE!r})
x = tok(["A test title" + tok.sep_token + "A short abstract about soil carbon."] * 2, return_tensors="pt", padding=True)
amp = torch.bfloat16 if {DEVICE!r} == "cpu" or torch.cuda.get_device_capability()[0] >= 8 else torch.float16
with torch.autocast({DEVICE!r}, dtype=amp):
    h = m(input_ids=x["input_ids"].to({DEVICE!r}), attention_mask=x["attention_mask"].to({DEVICE!r})).last_hidden_state
h.float().sum().backward()
print("ok", tuple(h.shape))
'''
    r = subprocess.run([sys.executable, '-c', code], capture_output=True, text=True,
                       env=dict(os.environ, CUDA_VISIBLE_DEVICES='0'))
    print(model, '→', 'працює' if r.returncode == 0 else 'ПОМИЛКА, пропускаю:\n' + r.stderr[-3000:])
    return r.returncode == 0

jobs = [j for j in JOBS if preflight(j[0])][:N_GPU]
assert jobs, 'Жодна модель не пройшла перевірку (див. помилки вище).'
procs = []
for gpu, (model, epochs, lr) in enumerate(jobs):
    log_path = W / f'log_{model.split("/")[-1]}.txt'
    log = open(log_path, 'w', encoding='utf-8')
    cmd = [sys.executable, '03_train_transformer.py', '--model', model, '--epochs', str(epochs), '--lr', lr,
           '--time-budget-h', str(BUDGET_H), '--device', DEVICE] + TRAIN_ARGS
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED='1', TQDM_MININTERVAL='60')
    procs.append((model, subprocess.Popen(cmd, cwd=W, env=env, stdout=log, stderr=subprocess.STDOUT), log, log_path))
    print(f'Запущено {model} на GPU {gpu}: {epochs} епохи, lr {lr}; журнал {log_path.name}')

In [ ]:
# 4. Чекаю, поки навчання закінчиться (стан друкується раз на 10 хвилин)
def last_line(path):
    try:
        lines = [l for l in path.read_text(encoding='utf-8', errors='replace').replace('\r', '\n').splitlines() if l.strip()]
        return lines[-1][:160] if lines else ''
    except FileNotFoundError:
        return ''

start = time.time()
while any(p.poll() is None for _, p, _, _ in procs):
    time.sleep(600 if DEVICE == 'cuda' else 5)
    for model, p, _, log_path in procs:
        state = 'працює' if p.poll() is None else f'завершено (код {p.returncode})'
        print(f'{(time.time() - start) / 3600:4.1f} год | {model}: {state} | {last_line(log_path)}', flush=True)
for model, p, log, log_path in procs:
    log.close()
    print(f'{model}: код завершення {p.returncode}' + ('' if p.returncode == 0 else f' — див. {log_path.name}'))

In [ ]:
# 5. Результати в Output: по архіву на модель, звіти, журнали
made, n_models = [], 0
for d in sorted((W / 'models').glob('*')):
    if (d / 'test_probs.npy').exists():
        shutil.make_archive(str(OUTPUT / d.name), 'zip', root_dir=W / 'models', base_dir=d.name)
        made.append(f'{d.name}.zip ({(OUTPUT / (d.name + ".zip")).stat().st_size / 2**30:.2f} ГБ)')
        n_models += 1
if (W / 'reports').exists():
    shutil.make_archive(str(OUTPUT / 'reports_kaggle'), 'zip', root_dir=W, base_dir='reports')
    made.append('reports_kaggle.zip')
for f in W.glob('log_*.txt'):
    shutil.copy(f, OUTPUT / f.name)
if n_models:
    shutil.rmtree(W)                              # копії даних і моделей в Output не потрібні
print('Output:', '; '.join(made) if made else 'моделей немає — див. журнали log_*.txt')